[Guide Index](README.md)

# 02d The Oxigraph-backed graph (`StarLayerGraph` / `StarLayerDataset`)

`StarLayerGraph` (one graph) and `StarLayerDataset` (a set of named graphs) each hold a `pyoxigraph.Store`, the Oxigraph database.

`StarLayerGraph` and `StarLayerDataset` have three kinds of methods:

- rdflib-style methods (`add`, `remove`, `triples`, `parse`, `serialize`, `query`, `update`, `bind`).
- other methods StarLayer adds (`infer`, `profile`, `skolemize`, ...).
- the Oxigraph store's own methods (`load`, `flush`, `backup`, ...), reached through `.ox`.

Every method of the Oxigraph store can be called through `.ox`, for example `g.ox.load(...)`, on the graph or the dataset. The methods can also be called on the store directly, reachable through `.store`. The preferred way is `.ox`. It records a change after a write, which is useful when StarLayer maintains caches of store data. 

Oxigraph documentation can be found here [pyoxigraph](https://pyoxigraph.readthedocs.io/).

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections reuse variables from earlier ones. Section 2 needs the OWL reasoners (`pip install starlayer-reasoning`, see [2.b](02b-graphs-inferencing.ipynb)) and section 3 needs rdflib (`pip install "starlayer[rdflib]"`). 

## 1. Calling store methods on a graph

All the graphs of a dataset share one store. A `StarLayerGraph` is one graph of it, and `.store` is the whole store (`as_store(x)` returns it for a wrapper and passes a `Store` through unchanged). There are three ways to load data into a graph. Here `people` gets one triple each way:

1. **rdflib style**: `parse`.
2. **The store's `load`, called through `.ox`**: Oxigraph's own method, with Oxigraph's arguments.
3. **The same `load`, called on `.store`**: the same arguments and the same result.


In [ ]:
import pyoxigraph as ox

from starlayer.graph import NamedNode, StarLayerDataset, as_store

EX = "http://example.org/"
ds = StarLayerDataset()
people = ds.graph(NamedNode(EX + "people"))

# 1. rdflib style
people.parse(data="<http://example.org/ann> <http://example.org/knows> <http://example.org/bob> .", format="turtle")
# 2. the store's load, called through .ox: the graph is named, as Oxigraph requires.
people.ox.load(b"<http://example.org/ann> <http://example.org/knows> <http://example.org/cara> .", ox.RdfFormat.TURTLE, to_graph=NamedNode(EX + "people"))
# 3. the same load, called on .store
people.store.load(b"<http://example.org/ann> <http://example.org/knows> <http://example.org/dan> .", ox.RdfFormat.TURTLE, to_graph=NamedNode(EX + "people"))

print("people:", len(people))

people: 3


Add a second graph, `places`, to the same store. A graph acts only on itself: asking `people` to load into `places` is a `ValueError`. 

Methods about the store as a whole such as `named_graphs` acts on the whole store, also when called on a graph, so `people.ox.named_graphs()` lists both graphs. 

In [2]:
places = ds.graph(NamedNode(EX + "places"))
places.ox.load(b"<http://example.org/oslo> <http://example.org/in> <http://example.org/norway> .", ox.RdfFormat.TURTLE, to_graph=NamedNode(EX + "places"))
print("people:", len(people), "places:", len(places), "| the store holds", len(as_store(people)), "quads, shared:", people.store is places.store)

try:
    people.ox.load(b"<http://example.org/x> <http://example.org/p> <http://example.org/y> .", ox.RdfFormat.TURTLE, to_graph=NamedNode(EX + "places"))
except ValueError as error:
    print("ValueError:", error)

print("named graphs, from a graph:", sorted(str(n) for n in people.ox.named_graphs()))


people: 3 places: 1 | the store holds 4 quads, shared: True
ValueError: this StarLayerGraph holds graph <http://example.org/people>; ox.load() cannot act on graph <http://example.org/places>
named graphs, from a graph: ['<http://example.org/people>', '<http://example.org/places>']


## 2. Write through the graph, not `.store`: the caching warning

StarLayer keeps some results between calls, and uses the record of changes to know when they are out of date.  For example, StarLayer caches entailed triples of `query(entailment=..., infer="changed")` and discards the cache when the graph changes.    A write made directly on `.store` is not registered as a change, so those results may go stale. Query entailment and the `infer=` option are covered in [3.a](03a-sparql-entailment.ipynb), section 3.

The example below caches the entailed graph of an OWL 2 RL query, then adds a fact directly on the store. `ex:y` is an `ex:A`, and every `ex:A` is an `ex:B`, but the query still says it is not a `ex:B`. Adding the same fact with `g.add`, the standard add, records the change, and the query is right.

To use Oxigraph's own `add`, which takes a quad, with the change recorded, call `g.ox.add(quad)`. `g.as_store()` hands you the store itself, with the data marked as changed first.

In [ ]:
from starlayer.graph import ENTAILMENT, RDF, StarLayerGraph

EXN = "http://example.org/"
g = StarLayerGraph()
g.parse(
    data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

    ex:A rdfs:subClassOf ex:B .
    """,
    format="turtle",
)


def y_is_a_b():
    query = "ASK { <http://example.org/y> a <http://example.org/B> }"
    return g.query(query, entailment=ENTAILMENT["OWL-RDF-Based"], infer="changed").askAnswer

y_is_an_a = (NamedNode(EXN + "y"), RDF.type, NamedNode(EXN + "A"))

print("y is a B:", y_is_a_b())                                   # the entailed graph is now cached
g.store.add(ox.Quad(*y_is_an_a, ox.DefaultGraph()))              # Oxigraph's add on .store: StarLayer is not told
print("after a write on .store:", y_is_a_b(), "(stale)")
g.add(y_is_an_a)                                                 # the standard add: StarLayer records the change
print("after a write through the graph:", y_is_a_b())

y is a B: False
after a write on .store: False (stale)
after a write through the graph: True


A method that writes through `.ox` (`add`, `extend`, `remove`, `clear`, `load`, `bulk_load`, `bulk_extend`, `add_graph`, `clear_graph`, `remove_graph`, `update`) also records that the data changed. The example above shows why that matters. A method that only reads (`query`, `quads_for_pattern`, `named_graphs`, ...) does not.

Managing graphs is also on the classes in their own form, the same as the `.ox` method: a dataset has `named_graphs()`, `add_graph(name)`, `clear_graph(name)` and `remove_graph(name)` (each takes a name or a graph), and a graph has `named_graphs()` and `remove_graph()`. SPARQL Update on a graph is aimed at that graph too: `people.update("CLEAR ALL")` clears `people` only, a triple in an `INSERT DATA` with no `GRAPH` goes into `people`, and an update that names another graph is a `ValueError`. `read_only` makes a store, so it is not under `.ox`: `StarLayerGraph.read_only(path)` (or `StarLayerDataset.read_only(path)`) opens an existing on-disk store for reading. `.ox` accepts only the names of Oxigraph's store methods; any other name is an `AttributeError`. Several methods are about files on disk: `flush` and `optimize` do nothing on an in-memory store, and `backup` raises an error. They are covered in [6.b](06b-backend-graph-databases.ipynb).

## 3. Converting to and from rdflib

A `StarLayerGraph` is not an `rdflib.Graph`. The methods have the same shape, but the terms that come out are pyoxigraph terms, and rdflib terms and graphs are not accepted. To convert a whole graph, use `StarLayerGraph.from_rdflib()` and `to_rdflib()`. Both need rdflib (`pip install "starlayer[rdflib]"`), which StarLayer imports only when a conversion runs. A triple term, which rdflib cannot hold, goes out in the RDF 1.1 basic encoding (section 5).

In [4]:
import rdflib

rdf_graph = rdflib.Graph()
rdf_graph.add((rdflib.URIRef(EX + "a"), rdflib.URIRef(EX + "knows"), rdflib.Literal(4)))

converted = StarLayerGraph.from_rdflib(rdf_graph)
print("in StarLayer:", [type(o).__module__ for o in converted.objects()], "| back in rdflib:", [type(o).__module__ for o in converted.to_rdflib().objects()])

in StarLayer: ['pyoxigraph'] | back in rdflib: ['rdflib.term']


## 4. What the store keeps of a literal

**The store rewrites some literals on insert** (issue #190). A lexical form such as `"04"^^xsd:integer` is stored as `4`; the original text is not recoverable from the store. Ill-formed values and strings are kept as written.

In [5]:
h = StarLayerGraph()
h.parse(data='<http://e/a> <http://e/age> "04"^^<http://www.w3.org/2001/XMLSchema#integer> .', format="turtle")
print([str(o) for o in h.objects()])

['"4"^^<http://www.w3.org/2001/XMLSchema#integer>']


## 5. Handing a graph to an RDF 1.1 consumer

**RDF 1.1 output.** The wrapper stores RDF 1.2 terms, including triple terms. To hand a graph to an RDF 1.1 consumer, serialise with `rdf_version="1.1"`: each triple term becomes the "basic encoding", a blank node typed `rdf:PropositionForm` with a subject, predicate and object property.

In [6]:
t = StarLayerGraph()
t.parse(data='''PREFIX ex: <http://example.org/>
ex:a ex:says <<( ex:b ex:knows ex:c )>> .''', format="turtle")
print(t.serialize(format="turtle", rdf_version="1.1", bnode_labels="short"))

@prefix ex: <http://example.org/> .
ex:a ex:says _:b0 .
_:b0 <http://www.w3.org/1999/02/22-rdf-syntax-ns#propositionFormObject> ex:c ;
	<http://www.w3.org/1999/02/22-rdf-syntax-ns#propositionFormPredicate> ex:knows ;
	<http://www.w3.org/1999/02/22-rdf-syntax-ns#propositionFormSubject> ex:b ;
	a <http://www.w3.org/1999/02/22-rdf-syntax-ns#PropositionForm> .



`starlayer.graph.interop` does the same conversion in memory, on quads. `to_basic(quads)` replaces each triple term with its basic encoding, `from_basic(quads)` finds that structure and rebuilds the triple term, and `has_triple_terms(quads)` says whether there is one to replace. `to_rdf11(graph)` also encodes base-direction literals (`direction_policy="datatype"`, `"error"` or `"drop"`), and `export_rdf11(...)` writes the result in an RDF 1.1 format.

In [7]:
from starlayer.graph import interop

quads = list(t.store)
print("has a triple term:", interop.has_triple_terms(quads))
basic = interop.to_basic(quads)
print(len(quads), "quad ->", len(basic), "quads in the basic encoding; a triple term left:", interop.has_triple_terms(basic))
restored = interop.from_basic(basic)
print("round trip restores it:", {(q.subject, q.predicate, q.object) for q in restored} == {(q.subject, q.predicate, q.object) for q in quads})

has a triple term: True
1 quad -> 5 quads in the basic encoding; a triple term left: False
round trip restores it: True
